# Objetivo: compor um prato com proteína, carboidrato e vegetal com a menor quantidade de calorias

In [ ]:
import time
from dataclasses import dataclass
from enum import Enum
from itertools import cycle
import random
import itertools

# ======== PARAMETROS ========
dias_cardapio = 5
orcamento_maximo_prato = 40
meta_calorias = 700
torelancia_caloria = 50
quantidade_ingreditens = 25

# ======== FUNCOES ========
class TipoIngrediente(Enum):
    PROTEINA = "proteina"
    CARBOIDRATO = "carboidrato"
    VEGETAL = "vegetal"


@dataclass
class Ingrediente:
    nome: str
    preco: float
    caloria: int
    tipo: TipoIngrediente 


def gerar_mocks_ingredientes(quantidade: int) -> list[Ingrediente]:
    tipos = list(TipoIngrediente)
    tipos_distribuidos = cycle(tipos)

    ingredientes = []

    for i in range(quantidade):
        tipo = next(tipos_distribuidos)

        if tipo == TipoIngrediente.CARBOIDRATO:
            caloria = random.randint(200, 450)

        elif tipo == TipoIngrediente.PROTEINA:
            caloria = random.randint(150, 400)

        else:
            caloria = random.randint(20, 150)

        ingrediente = Ingrediente(
            nome=f"Ingrediente {i + 1}",
            preco=round(random.uniform(2.0, 30.0), 2),
            caloria=caloria,
            tipo=tipo
        )

        ingredientes.append(ingrediente)

    return ingredientes


def otimizar_lista_otima(ingredientes: list[Ingrediente]):
    iteracao = 0
    tempo_inicio = time.time()
    combinacoes = itertools.product([0, 1], repeat=quantidade_ingreditens)

    combinacoes_possiveis = []

    for combinacao in combinacoes:
        iteracao += 1
        if combinacao.count(1) != 3:
            continue

        calorias = 0
        preco = 0
        tipos_utilizados = []
        ingredientes_selecionados = []
        for i in range(quantidade_ingreditens):
            iteracao += 1
            if combinacao[i] == 1:
                calorias += ingredientes[i].caloria
                preco += ingredientes[i].preco
                tipos_utilizados.append(ingredientes[i].tipo)
                ingredientes_selecionados.append(ingredientes[i])

        if not TipoIngrediente.VEGETAL in tipos_utilizados:
            continue

        if not TipoIngrediente.CARBOIDRATO in tipos_utilizados:
            continue

        if not TipoIngrediente.PROTEINA in tipos_utilizados:
            continue

        if not meta_calorias - torelancia_caloria <= calorias <= meta_calorias + torelancia_caloria:
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": ingredientes_selecionados,
            "calorias": calorias,
            "preco": preco
        }

        combinacoes_possiveis.append(combinacao_info)

    if len(combinacoes_possiveis) < dias_cardapio:
        return None

    combinacoes_possiveis.sort(key=lambda combinacao: combinacao["preco"])

    tempo_fim = time.time() - tempo_inicio


    return {
        "tempo": tempo_fim,
        "iteracoes": iteracao,
        "pratos": combinacoes_possiveis[0: dias_cardapio]
    }



def otimizar_lista_heuristico(ingredientes: list[Ingrediente]):
    tempo_inicio = time.time()
    iteracoes = 0
    caloria_ideal_carbo = meta_calorias * 0.5
    caloria_ideal_proteina = meta_calorias * 0.4
    caloria_ideal_vegetal = meta_calorias * 0.1

    ingredientes_candidatos = {
        "carbo": [],
        "proteina": [], 
        "vegetal": []
    }

    for ingrediente in ingredientes:
        iteracoes += 1
        if (
            ingrediente.tipo == TipoIngrediente.VEGETAL
            and caloria_ideal_vegetal - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_vegetal + torelancia_caloria
        ):
            ingredientes_candidatos["vegetal"].append(ingrediente)

        if (
            ingrediente.tipo == TipoIngrediente.CARBOIDRATO
            and caloria_ideal_carbo - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_carbo + torelancia_caloria
        ):
            ingredientes_candidatos["carbo"].append(ingrediente)

        if (
            ingrediente.tipo == TipoIngrediente.PROTEINA
            and caloria_ideal_proteina - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_proteina + torelancia_caloria
        ):
            ingredientes_candidatos["proteina"].append(ingrediente)

    if (
        not ingredientes_candidatos["carbo"]
        or not ingredientes_candidatos["proteina"]
        or not ingredientes_candidatos["vegetal"]
    ):
        return None

    combinacoes = itertools.product(
        ingredientes_candidatos["carbo"],
        ingredientes_candidatos["proteina"],
        ingredientes_candidatos["vegetal"]
    )

    pratos_candidatos = []

    for carbo, proteina, vegetal in combinacoes:
        iteracoes += 1
        calorias = carbo.caloria + proteina.caloria + vegetal.caloria
        preco = carbo.preco + proteina.preco + vegetal.preco

        if not (
            meta_calorias - torelancia_caloria
            <= calorias
            <= meta_calorias + torelancia_caloria
        ):
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": [carbo, proteina, vegetal],
            "calorias": calorias,
            "preco": preco
        }

        pratos_candidatos.append(combinacao_info)

    if len(pratos_candidatos) < dias_cardapio:
        return None

    pratos_candidatos.sort(
        key=lambda combinacao: combinacao["preco"]
    )

    tempo_fim = time.time() - tempo_inicio



    return {
        "tempo": tempo_fim,
        "iteracoes": iteracoes,
        "pratos": pratos_candidatos[:dias_cardapio]
    }

        
def calcular_gap(lista_exaustiva, lista_heuristica):
    if lista_exaustiva is None or lista_heuristica is None:
        return None

    custo_exaustivo = sum(
        prato["preco"] for prato in lista_exaustiva["pratos"]
    )

    custo_heuristico = sum(
        prato["preco"] for prato in lista_heuristica["pratos"]
    )

    if custo_exaustivo == 0:
        return 0

    gap = ((custo_heuristico - custo_exaustivo) / custo_exaustivo) * 100

    return {
        "custo_exaustivo": round(custo_exaustivo, 2),
        "custo_heuristico": round(custo_heuristico, 2),
        "gap": round(gap, 2)
    }


def exibir_resultados(nome: str, resultado):
    print(f"\n{'=' * 60}")
    print(f"{nome:^60}")
    print(f"{'=' * 60}")

    if resultado is None:
        print("Nenhuma solução válida encontrada.")
        return

    print(f"Tempo de execução: {resultado['tempo']:.6f} segundos")
    print(f"Iterações: {resultado['iteracoes']}")
    print(f"Quantidade de pratos: {len(resultado['pratos'])}")

    print(f"\n{'-' * 60}")
    print("PRATOS")
    print(f"{'-' * 60}")

    custo_total = 0

    for numero, prato in enumerate(resultado["pratos"], start=1):
        print(f"\nPrato {numero}")

        for ingrediente in prato["itens"]:
            print(
                f"  - {ingrediente.nome:<15} "
                f"| {ingrediente.tipo.value:<12} "
                f"| {ingrediente.caloria:>3} kcal "
                f"| R$ {ingrediente.preco:>6.2f}"
            )

        print(f"  Calorias totais: {prato['calorias']} kcal")
        print(f"  Preço total:     R$ {prato['preco']:.2f}")

        custo_total += prato["preco"]

    print(f"\n{'-' * 60}")
    print(f"Custo total do cardápio: R$ {custo_total:.2f}")

        
def exibir_gap(resultado_gap):
    print(f"\n{'=' * 60}")
    print(f"{'COMPARAÇÃO DAS SOLUÇÕES':^60}")
    print(f"{'=' * 60}")

    if resultado_gap is None:
        print("Não foi possível calcular o GAP.")
        return

    print(
        f"Custo solução exaustiva: R$ "
        f"{resultado_gap['custo_exaustivo']:.2f}"
    )

    print(
        f"Custo solução heurística: R$ "
        f"{resultado_gap['custo_heuristico']:.2f}"
    )

    print(f"GAP: {resultado_gap['gap']:.2f}%")

# ======== EXECUÇÃO ========
ingredientes = gerar_mocks_ingredientes(quantidade_ingreditens)

cardapio_exaustivo = otimizar_lista_otima(ingredientes)
cardapio_heuristico = otimizar_lista_heuristico(ingredientes)

gap = calcular_gap(
    cardapio_exaustivo,
    cardapio_heuristico
)

exibir_resultados(
    "SOLUÇÃO EXAUSTIVA",
    cardapio_exaustivo
)

exibir_resultados(
    "SOLUÇÃO HEURÍSTICA",
    cardapio_heuristico
)

exibir_gap(gap)


                     SOLUÇÃO EXAUSTIVA                      
Tempo de execução: 8.787053 segundos
Iterações: 33611932
Quantidade de pratos: 5

------------------------------------------------------------
PRATOS
------------------------------------------------------------

Prato 1
  - Ingrediente 4   | proteina     | 316 kcal | R$   6.96
  - Ingrediente 11  | carboidrato  | 266 kcal | R$   4.53
  - Ingrediente 24  | vegetal      |  91 kcal | R$   6.64
  Calorias totais: 673 kcal
  Preço total:     R$ 18.13

Prato 2
  - Ingrediente 8   | carboidrato  | 412 kcal | R$   9.67
  - Ingrediente 22  | proteina     | 177 kcal | R$   3.30
  - Ingrediente 24  | vegetal      |  91 kcal | R$   6.64
  Calorias totais: 680 kcal
  Preço total:     R$ 19.61

Prato 3
  - Ingrediente 7   | proteina     | 201 kcal | R$   3.37
  - Ingrediente 8   | carboidrato  | 412 kcal | R$   9.67
  - Ingrediente 24  | vegetal      |  91 kcal | R$   6.64
  Calorias totais: 704 kcal
  Preço total:     R$ 19.68

Prato 4
 